# Chapter 5: Pretraining on Unlabeled Data 🔥

The "soul injection" chapter — training random weights into a meaningful language model.

We'll pretrain a small GPT model on "The Verdict" by Edith Wharton (~20K tokens).

**What to watch for:**
- Loss curve: should decrease steadily
- Perplexity: should drop from ~50K (random) to single digits
- Generated text: from gibberish → coherent English

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import tiktoken
import matplotlib.pyplot as plt

from ch02.dataloader import create_dataloader
from ch04.config import GPTConfig
from ch04.gpt_model import GPTModel
from ch05.train import train_model
from ch05.evaluate import calc_loss_loader, calc_perplexity
from ch05.utils import generate_text

torch.manual_seed(42)
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Load Data & Create Dataloaders

In [ ]:
# Load "The Verdict" text
with open('../ch02/the-verdict.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokenizer = tiktoken.get_encoding('gpt2')
total_tokens = len(tokenizer.encode(text))
print(f'Text length: {len(text):,} characters')
print(f'Total tokens: {total_tokens:,}')

# Split: 90% train, 10% val
split_idx = int(len(text) * 0.9)
train_text = text[:split_idx]
val_text = text[split_idx:]

MAX_LEN = 256
BATCH_SIZE = 4

train_loader = create_dataloader(
    train_text, tokenizer,
    max_len=MAX_LEN, stride=MAX_LEN,  # non-overlapping
    batch_size=BATCH_SIZE, shuffle=True,
)
val_loader = create_dataloader(
    val_text, tokenizer,
    max_len=MAX_LEN, stride=MAX_LEN,
    batch_size=BATCH_SIZE, shuffle=False,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')

## 2. Initialize Small GPT Model

In [ ]:
# Small config for fast training on CPU
config = GPTConfig(
    vocab_size=50257,
    d_model=128,       # small (GPT-2 uses 768)
    n_heads=4,         # fewer (GPT-2 uses 12)
    n_layers=4,        # fewer (GPT-2 uses 12)
    max_seq_len=256,   # shorter (GPT-2 uses 1024)
    dropout=0.1,
)

model = GPTModel(config)
model = model.to(device)
print(f'Parameters: {model.count_parameters():,}')
print(f'Config: d_model={config.d_model}, n_heads={config.n_heads}, n_layers={config.n_layers}')

## 3. Text Generation BEFORE Training

With random weights, the model generates complete gibberish.
This is our "before" snapshot.

In [ ]:
prompt = 'Every effort moves you'

before_text = generate_text(
    model, prompt,
    max_new_tokens=50,
    temperature=1.0, top_k=50,
    device=device,
)

print(f'Prompt: "{prompt}"')
print(f'Generated (random weights):')
print(f'  "{before_text}"')

# Calculate initial loss
model.to(device)
init_train_loss = calc_loss_loader(model, train_loader, device, max_batches=5)
init_val_loss = calc_loss_loader(model, val_loader, device)
print(f'\nInitial train loss: {init_train_loss:.4f} (PPL: {calc_perplexity(init_train_loss):.0f})')
print(f'Initial val loss:   {init_val_loss:.4f} (PPL: {calc_perplexity(init_val_loss):.0f})')

## 4. Pretrain the Model

This should take a few minutes on CPU, faster on MPS/CUDA.

In [ ]:
history = train_model(
    model, train_loader, val_loader,
    num_epochs=15,
    max_lr=5e-4,
    min_lr=1e-5,
    warmup_frac=0.1,
    weight_decay=0.1,
    max_grad_norm=1.0,
    eval_every=10,
    sample_every=50,
    checkpoint_dir='checkpoints',
    device=device,
    prompt=prompt,
)

## 5. Loss Curves (Train vs Val)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Loss curves
axes[0].plot(history['steps'], history['train_losses'], label='Train', linewidth=2)
axes[0].plot(history['steps'], history['val_losses'], label='Val', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Perplexity
axes[1].plot(history['steps'], history['perplexities'], color='green', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity')
axes[1].set_yscale('log')  # Log scale because PPL starts very high
axes[1].grid(True, alpha=0.3)

# Plot 3: Learning rate schedule
axes[2].plot(history['steps'], history['lrs'], color='orange', linewidth=2)
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 6. Text Generation AFTER Training

After training on ~20K tokens, the model should generate coherent English
(likely reproducing patterns from "The Verdict").

In [ ]:
prompts = [
    'Every effort moves you',
    'The artist',
    'She had been',
]

print('=' * 60)
print('Text Generation After Training')
print('=' * 60)

for p in prompts:
    generated = generate_text(
        model, p,
        max_new_tokens=80,
        temperature=0.8, top_k=25,
        device=device,
    )
    print(f'\nPrompt: "{p}"')
    print(f'Output: "{generated}"')
    print('-' * 60)

## 7. Before vs After Comparison

In [ ]:
after_text = generate_text(
    model, prompt,
    max_new_tokens=50,
    temperature=0.8, top_k=25,
    device=device,
)

print('=' * 60)
print('BEFORE Training (random weights):')
print(f'  "{before_text}"')
print()
print('AFTER Training (15 epochs on The Verdict):')
print(f'  "{after_text}"')
print('=' * 60)

# Final metrics
final_train_loss = history['train_losses'][-1]
final_val_loss = history['val_losses'][-1]
print(f'\nFinal train loss: {final_train_loss:.4f}')
print(f'Final val loss:   {final_val_loss:.4f}')
print(f'Final perplexity: {history["perplexities"][-1]:.1f}')
print(f'\nInitial → Final loss: {init_val_loss:.2f} → {final_val_loss:.2f}')
print(f'Initial → Final PPL:  {calc_perplexity(init_val_loss):.0f} → {history["perplexities"][-1]:.1f}')

## Key Takeaways

1. **Random → Meaningful**: With just ~20K tokens and a few minutes of training, the model goes from gibberish to coherent English
2. **Overfitting is expected**: On such small data, the model memorizes the training text — that's fine for a demo
3. **Loss drops fast**: Most learning happens in the first few epochs
4. **LR schedule matters**: Warmup prevents early instability, cosine decay helps convergence
5. **Scale is everything**: Real LLMs train on billions of tokens for weeks on GPU clusters